# SeizureGuard — reproduction complète du pipeline scientifique sur Google Colab

Ce notebook **rejoue** la pipeline de bout en bout — preprocessing SeizeIT2, validation Leave-One-Subject-Out, entraînement MLP TinyML, estimation Edge AI ESP32 — et reproduit les chiffres publiés du projet du Groupe 2 SUP'COM.

## En quoi est-ce différent du notebook `launch_streamlit_colab.ipynb` ?

| Notebook | Rôle | Données | Temps |
|---|---|---|---|
| `launch_streamlit_colab.ipynb` | Lance le **portfolio Streamlit** en ligne via Cloudflare. Lecture seule des résultats déjà calculés. | Aucune — utilise les CSV/JSON déjà dans le repo. | ~1 min |
| `reproduce_pipeline_colab.ipynb` (ce notebook) | **Recalcule** les résultats à partir des EDF bruts. Vérifie que les chiffres du portfolio sont bien reproductibles. | SeizeIT2 (~quelques GB pour 6 sujets). | 20–60 min selon les sujets. |

## Chiffres cibles à reproduire (référence : `results/multirun_loso_pooled.json`)

| Modèle | Recall pooled | TP / N_pos | Accuracy pooled |
|---|---|---|---|
| Decision Tree | 11,0 % | 98 / 893 | 94,5 % |
| SVM RBF | 6,5 % | 58 / 893 | 96,2 % |
| Random Forest | 3,25 % | 29 / 893 | 97,4 % |
| MLP TinyML INT8 | 8,7 % | 78 / 893 | 91,6 % |

Dataset : 33 925 fenêtres test, 893 positives, baseline « tout négatif » ≈ 97,37 %.

Si vous obtenez exactement ces TP/N_pos, la reproduction est validée.

## 1. Cloner le dépôt et installer les dépendances pipeline

In [ ]:
REPO_URL = "https://github.com/akiroussama/iot-edge-ai-seizure-detection.git"
REPO_DIR = "iot-edge-ai-seizure-detection"
BRANCH = "main"

from pathlib import Path

if Path(REPO_DIR).exists():
    !rm -rf {REPO_DIR}

!git clone --depth 1 -b {BRANCH} {REPO_URL}
%cd {REPO_DIR}

!ls src/

In [ ]:
# Dépendances de la pipeline (mne, scikit-learn, scipy, matplotlib, etc.)
!pip install -q --upgrade pip
!pip install -q -r requirements-pipeline.txt

# openneuro-py est nécessaire seulement si on choisit l'option B (téléchargement OpenNeuro).
# Coût : quelques secondes, donc on le préinstalle.
!pip install -q openneuro-py

import mne, sklearn, numpy, scipy, matplotlib
print("mne          :", mne.__version__)
print("scikit-learn :", sklearn.__version__)
print("numpy        :", numpy.__version__)
print("scipy        :", scipy.__version__)
print("matplotlib   :", matplotlib.__version__)

## 2. Acquisition des données SeizeIT2

Les scripts `src/` attendent la structure suivante :

```
data/
├── sub-001_run-07/
│   ├── eeg.edf
│   ├── mov.edf
│   └── events.tsv
├── sub-032_run-XX/
│   └── ...
├── sub-085_run-XX/
├── sub-096_run-XX/
├── sub-124_run-XX/
└── sub-125_run-XX/
```

Une seule structure cible, mais deux façons d'y arriver :

- **Option A — Google Drive** (recommandée, rapide).  Si vous avez déjà préparé `data/` sur votre Drive, on monte le Drive et on copie. Quelques minutes.
- **Option B — OpenNeuro automatique** (lente, plusieurs GB). Le notebook télécharge les EDF + events.tsv depuis OpenNeuro `ds005873` puis aplatit la structure BIDS dans le layout attendu.

**Exécutez UNE seule des deux options.**

### Option A — Monter un Google Drive contenant déjà `data/`

Pré-requis : sur votre Drive, créer un dossier `SeizeIT2_subset/` qui contient déjà `sub-001_run-07/`, `sub-032_run-XX/`, etc., avec les trois fichiers attendus.

In [ ]:
# Décommentez le bloc ci-dessous si vous voulez utiliser l'option A.

# from google.colab import drive
# drive.mount('/content/drive')

# DRIVE_SRC = '/content/drive/MyDrive/SeizeIT2_subset'
# !mkdir -p data
# !rsync -av --progress {DRIVE_SRC}/ data/

# !ls data/

### Option B — Téléchargement automatique depuis OpenNeuro

Stratégie minimaliste : pour chaque sujet, on télécharge tous les `events.tsv` (très petits) ; on identifie les runs qui contiennent au moins une crise (`eventType` commençant par `sz`) ; on télécharge **uniquement** les EDF de ces runs ; puis on aplatit en `data/sub-XXX_run-YY/`.

Coût attendu : quelques GB de bande passante, 15–40 min selon les sujets.

**Note :** OpenNeuro est public mais ses serveurs peuvent throttler. Si une cellule échoue avec un timeout, relancez-la, le téléchargement est resumable.

In [ ]:
import openneuro
import shutil
from pathlib import Path

DATASET = "ds005873"
SUBJECTS = ["001", "032", "085", "096", "124", "125"]
WORK = Path("openneuro_raw")
WORK.mkdir(parents=True, exist_ok=True)

# Étape 1 — télécharger les sidecars + events (très léger, < 5 MB total).
include_meta = []
for sub in SUBJECTS:
    include_meta.extend([
        f"sub-{sub}/*events.tsv",
        f"sub-{sub}/*.json",
    ])

print("Téléchargement des métadonnées (events.tsv) pour identifier les runs avec crise...")
openneuro.download(
    dataset=DATASET,
    target_dir=str(WORK),
    include=include_meta,
)
print("OK.")

In [ ]:
# Étape 2 — pour chaque sujet, identifier les events.tsv qui contiennent au moins
# une crise et choisir un seul run par sujet (le premier qui en contient).
import csv
import re

RUN_RE = re.compile(r"run-(\d+)")
selected = {}  # subject -> (run_id, events_path)

for sub in SUBJECTS:
    sub_dir = WORK / f"sub-{sub}"
    if not sub_dir.exists():
        print(f"sub-{sub} : aucun fichier téléchargé")
        continue
    for ev_path in sorted(sub_dir.rglob("*events.tsv")):
        run_m = RUN_RE.search(ev_path.name)
        if not run_m:
            continue
        with ev_path.open(encoding="utf-8") as f:
            rows = list(csv.DictReader(f, delimiter="\t"))
        has_sz = any(r.get("eventType", "").startswith("sz")
                     and r.get("eventType") not in ("bckg", "impd")
                     for r in rows)
        if has_sz:
            selected[sub] = (run_m.group(1), ev_path)
            break

for sub, (run, p) in selected.items():
    print(f"sub-{sub} -> run-{run}")
if len(selected) < len(SUBJECTS):
    missing = set(SUBJECTS) - set(selected)
    print(f"ATTENTION : runs introuvables pour {sorted(missing)}")

In [ ]:
# Étape 3 — télécharger UNIQUEMENT les EDF des runs sélectionnés (eeg.edf + mov.edf).
include_edf = []
for sub, (run, _) in selected.items():
    include_edf.extend([
        f"sub-{sub}/*run-{run}*_eeg.edf",
        f"sub-{sub}/*run-{run}*_mov.edf",
    ])

print(f"Téléchargement des EDF pour {len(selected)} sujets (~quelques GB)...")
openneuro.download(
    dataset=DATASET,
    target_dir=str(WORK),
    include=include_edf,
)
print("OK.")

In [ ]:
# Étape 4 — aplatir le layout BIDS en data/sub-XXX_run-YY/{eeg,mov}.edf + events.tsv.
data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)

for sub, (run, ev_path) in selected.items():
    target = data_dir / f"sub-{sub}_run-{run}"
    target.mkdir(parents=True, exist_ok=True)

    # events.tsv
    shutil.copy2(ev_path, target / "events.tsv")

    # eeg.edf et mov.edf : retrouver dans le dossier BIDS
    sub_dir = WORK / f"sub-{sub}"
    eeg_candidates = list(sub_dir.rglob(f"*run-{run}*_eeg.edf"))
    mov_candidates = list(sub_dir.rglob(f"*run-{run}*_mov.edf"))
    if not eeg_candidates or not mov_candidates:
        print(f"sub-{sub} run-{run} : EDF manquant, skip")
        continue
    shutil.copy2(eeg_candidates[0], target / "eeg.edf")
    shutil.copy2(mov_candidates[0], target / "mov.edf")
    print(f"OK : {target}")

!ls data/

## 3. Vérifier la structure des données

In [ ]:
from pathlib import Path
import re

DIR_RE = re.compile(r"^sub-(\d+)_run-(\d+)$")
required = {"eeg.edf", "mov.edf", "events.tsv"}

data_root = Path("data")
if not data_root.exists():
    raise SystemExit("data/ est absent. Exécute l'option A ou B au-dessus.")

runs_found = 0
for d in sorted(data_root.iterdir()):
    if not d.is_dir() or not DIR_RE.match(d.name):
        continue
    present = {p.name for p in d.iterdir()}
    missing = required - present
    ok = "OK" if not missing else f"MANQUE {missing}"
    sizes = {p.name: p.stat().st_size for p in d.iterdir() if p.name in required}
    print(f"{d.name:24} {ok}  sizes={sizes}")
    runs_found += int(not missing)

print(f"\nRuns valides : {runs_found}")

## 4. Preprocessing + extraction de features

`src/pipeline_multirun.py` lit chaque dossier `data/sub-*_run-*/`, applique le filtrage Butterworth, aligne EEG (256 Hz) et IMU (25 Hz) à 50 Hz, génère des fenêtres glissantes de 2,56 s avec 50 % de chevauchement, et calcule 80 features (variance, skewness, kurtosis, Higuchi Fractal Dimension, Spectral Entropy, statistiques IMU).

Sortie : `data/multirun_features.npz` (matrice unifiée X, y, subject_ids).

In [ ]:
!python src/pipeline_multirun.py

## 5. Entraînement LOSO classique : Decision Tree, SVM RBF, Random Forest

Pour chaque sujet S : train sur les 5 autres, test sur S. On accumule TP, FP, FN, TN par fold puis on calcule recall **pooled** (`ΣTP / Σ(TP+FN)`).

In [ ]:
!python src/train_multirun.py

## 6. Entraînement LOSO du MLP TinyML

Architecture 80 → 32 → 16 → 1, quantification INT8 simulée pour estimer la mémoire ESP32. Sortie : `results/mlp_loso.csv` + `results/mlp_loso_summary.json`.

In [ ]:
!python src/train_mlp.py

## 7. Estimation Edge AI ESP32 (RAM, latence, énergie)

In [ ]:
!python src/estimate_esp32_cost.py

## 8. Calcul des métriques pooled + recompute final

`src/compute_pooled_metrics.py` lit `results/multirun_loso.csv` et `results/mlp_loso.csv`, agrège micro/pooled, et écrit `results/multirun_loso_pooled.json`. C'est ce JSON qui est consommé par le portfolio Streamlit et le site Vercel.

In [ ]:
!python src/compute_pooled_metrics.py 2>/dev/null || echo "(script optionnel — peut ne pas exister sur toutes les versions)"

## 9. Génération des figures

In [ ]:
!python src/make_figures.py

from IPython.display import Image, display
from pathlib import Path

fig_dir = Path("results/figures")
for fig in sorted(fig_dir.glob("*.png")):
    print(fig.name)
    display(Image(str(fig)))

## 10. Comparer aux chiffres canoniques

On charge `results/multirun_loso_pooled.json` (recalculé à l'instant) et on le compare à la table de référence. Si la reproduction est correcte, les `tp` doivent matcher exactement (29 RF, 78 MLP, 98 DT, 58 SVM).

In [ ]:
import json
from pathlib import Path

CANONICAL = {
    "decision_tree":   {"tp":  98, "recall": 0.1097},
    "svm_rbf":         {"tp":  58, "recall": 0.0649},
    "random_forest":   {"tp":  29, "recall": 0.0325},
    "mlp_80_32_16_1":  {"tp":  78, "recall": 0.0873},
}

p = Path("results/multirun_loso_pooled.json")
if not p.exists():
    print("results/multirun_loso_pooled.json absent — la pipeline n'a pas fini correctement.")
else:
    pooled = json.loads(p.read_text(encoding="utf-8"))
    print(f"{'model':20} {'TP':>4} {'TP_canon':>8} {'rec':>8} {'rec_canon':>10} {'verdict':>10}")
    print("-" * 70)
    for m, info in pooled.get("models", {}).items():
        tp = info.get("tp", -1)
        rec = info.get("recall_pooled", -1.0)
        ref = CANONICAL.get(m)
        if ref is None:
            verdict = "unknown"
        elif tp == ref["tp"] and abs(rec - ref["recall"]) < 0.01:
            verdict = "MATCH"
        else:
            verdict = "DRIFT"
        ref_tp = ref["tp"] if ref else "-"
        ref_rec = ref["recall"] if ref else float("nan")
        print(f"{m:20} {tp:>4} {ref_tp:>8} {rec:>8.4f} {ref_rec:>10.4f} {verdict:>10}")

## 11. Conclusion

Si la dernière cellule affiche **MATCH** pour les 4 modèles, la reproduction scientifique est validée à TP près : les chiffres publiés dans le portfolio (RF 3,25 %, MLP 8,7 %, etc.) sont reproductibles à partir des EDF bruts de SeizeIT2.

Si certaines lignes affichent **DRIFT**, vérifier dans l'ordre :

1. Le nombre de sujets/runs effectivement traités (cf. cellule §3).
2. Les versions des dépendances (cf. cellule §1) — `mne 1.6+`, `scikit-learn 1.3+`.
3. Que le `random_state=42` n'a pas été modifié dans `src/train_multirun.py` et `src/train_mlp.py`.
4. Que les EDF téléchargés correspondent bien aux mêmes runs que ceux utilisés pour la publication initiale (les runs peuvent varier selon la sélection initiale — la pipeline reste valide, seuls les chiffres exacts changent).

Pour étendre l'expérience :

- Ajouter plus de sujets SeizeIT2 dans `data/` puis relancer la cellule §4.
- Modifier les hyperparamètres dans `src/train_mlp.py` (taille couches cachées, learning rate) pour explorer le compromis recall / mémoire.
- Mesurer le coût Edge AI réel sur un ESP32 instrumenté (INA219) et comparer à `results/esp32_cost_estimate.json`.